# LLM Proxy Suggestions For Active Learning

This notebook reads the active-learning annotation batch, sends each article to the UvA LLM proxy, and saves English translations plus LLM label suggestions for human review.

It is designed to be resumable: if the output file already exists, rows with completed suggestions are skipped.

## Setup

Before running, set your API key in the shell that launches Jupyter:

```bash
export LLMPROXY_API_KEY="your-key-here"
```

If needed, install the OpenAI client:

```bash
pip install openai
```

In [ ]:
from pathlib import Path
import json
import os
import re
import time

import pandas as pd
from tqdm.auto import tqdm

from openai import OpenAI

from config import LLMPROXY_API_KEY, LLMPROXY_BASE_URL, LLMPROXY_MODEL

In [ ]:
AL_DIR = Path("/home/akroon/data/1t_storage/RESPOND-victims-of-corruption/political_corruption_pipeline/active_learning")
INPUT_PATH = AL_DIR / "active_learning_batch_for_annotation.csv"
OUTPUT_PATH = AL_DIR / "active_learning_batch_with_llm_suggestions.csv"

MAX_CHARS = 6000
SAVE_EVERY = 10
SLEEP_BETWEEN_CALLS = 0.1

if not LLMPROXY_API_KEY:
    raise ValueError("Set LLMPROXY_API_KEY in your environment before running this notebook.")

client = OpenAI(api_key=LLMPROXY_API_KEY, base_url=LLMPROXY_BASE_URL)

print(f"Input:  {INPUT_PATH}")
print(f"Output: {OUTPUT_PATH}")
print(f"Model:  {LLMPROXY_MODEL}")

In [ ]:
al_df = pd.read_csv(INPUT_PATH)
print(f"Loaded {len(al_df):,} active-learning rows.")
display(al_df.head())
display(al_df["country"].value_counts().to_frame("rows"))
display(al_df["al_bucket"].value_counts().to_frame("rows"))

## Prompt And Parsing Helpers

The LLM output is a suggestion only. The final training label should be the human-reviewed label.

In [ ]:
def build_annotation_prompt(article_text: str) -> str:
    return f"""
You are helping annotate multilingual news articles for a research project.

Tasks:
1. Translate the article into clear, high-quality English.
2. Suggest whether the article is primarily about political corruption.

Definition:
Political corruption involves public officials or political decision-makers misusing political power for personal, political, or party gain.

Label rules:
- Use "political corruption" when political/public decision-makers are centrally involved in corruption, bribery, kickbacks, embezzlement of public funds, nepotism/cronyism, conflicts of interest, vote buying, abuse of office, or similar misconduct.
- Use "no political corruption" for private fraud, ordinary crime, business misconduct, general scandals, policing/military misconduct, or corruption mentioned only generically without political/public decision-makers being central.
- Use "unclear" when the article is too ambiguous or lacks enough information.

Return valid JSON only, with these keys:
{{
  "translated_text": "...",
  "llm_label_suggestion": "political corruption" | "no political corruption" | "unclear",
  "llm_confidence": 0-100,
  "llm_rationale": "short explanation",
  "llm_evidence": "short quote or paraphrase of key evidence"
}}

Article:
{article_text}
""".strip()


def extract_json(text: str) -> dict:
    text = text.strip()
    text = re.sub(r"^```(?:json)?", "", text).strip()
    text = re.sub(r"```$", "", text).strip()
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        match = re.search(r"\{.*\}", text, flags=re.DOTALL)
        if match:
            return json.loads(match.group(0))
        raise


def llm_translate_and_suggest(article_text: str, max_chars: int = MAX_CHARS) -> dict:
    article_text = "" if not isinstance(article_text, str) else article_text[:max_chars]
    if not article_text.strip():
        return {
            "translated_text": "",
            "llm_label_suggestion": "no political corruption",
            "llm_confidence": 0,
            "llm_rationale": "No content.",
            "llm_evidence": "",
        }

    response = client.chat.completions.create(
        model=LLMPROXY_MODEL,
        messages=[{"role": "user", "content": build_annotation_prompt(article_text)}],
        temperature=0,
    )
    raw = response.choices[0].message.content
    parsed = extract_json(raw)

    return {
        "translated_text": parsed.get("translated_text", ""),
        "llm_label_suggestion": parsed.get("llm_label_suggestion", ""),
        "llm_confidence": parsed.get("llm_confidence", ""),
        "llm_rationale": parsed.get("llm_rationale", ""),
        "llm_evidence": parsed.get("llm_evidence", ""),
    }

## Test A Few Rows First

Run this before the full batch. Inspect whether the translations and labels look sensible.

In [ ]:
test_rows = al_df.sample(min(3, len(al_df)), random_state=42)

for _, row in test_rows.iterrows():
    print("=" * 100)
    print(row.get("country", ""), row.get("uri", ""), row.get("al_bucket", ""))
    print("Model probability:", row.get("prob_political_corruption", ""))
    result = llm_translate_and_suggest(row.get("article_text", ""))
    print(json.dumps(result, ensure_ascii=False, indent=2)[:4000])

## Full Batch With Checkpointing

Set `RUN_FULL = True` only after the test rows look good. The notebook saves a checkpoint every `SAVE_EVERY` rows and resumes by `uri` if the output file already exists.

In [ ]:
RUN_FULL = False

if not RUN_FULL:
    print("RUN_FULL is False. Set it to True when you are ready to process the full batch.")
else:
    if OUTPUT_PATH.exists():
        existing = pd.read_csv(OUTPUT_PATH)
        done_uris = set(existing["uri"].dropna().astype(str)) if "uri" in existing.columns else set()
        print(f"Resuming from {OUTPUT_PATH}; already done: {len(done_uris):,}")
    else:
        existing = pd.DataFrame()
        done_uris = set()

    new_rows = []

    for _, row in tqdm(al_df.iterrows(), total=len(al_df), desc="LLM translation + suggestions"):
        uri = str(row.get("uri", ""))
        if uri in done_uris:
            continue

        out = row.to_dict()
        try:
            suggestion = llm_translate_and_suggest(row.get("article_text", ""))
            out.update(suggestion)
            out["llm_error"] = ""
        except Exception as exc:
            out.update({
                "translated_text": "",
                "llm_label_suggestion": "",
                "llm_confidence": "",
                "llm_rationale": "",
                "llm_evidence": "",
                "llm_error": repr(exc),
            })

        if "human_final_label" not in out:
            out["human_final_label"] = ""
        if "human_notes" not in out:
            out["human_notes"] = ""

        new_rows.append(out)

        if len(new_rows) % SAVE_EVERY == 0:
            checkpoint = pd.concat([existing, pd.DataFrame(new_rows)], ignore_index=True)
            checkpoint.to_csv(OUTPUT_PATH, index=False)
            print(f"Saved checkpoint: {OUTPUT_PATH} ({len(checkpoint):,} rows)")
            time.sleep(SLEEP_BETWEEN_CALLS)

    final = pd.concat([existing, pd.DataFrame(new_rows)], ignore_index=True)
    final.to_csv(OUTPUT_PATH, index=False)
    print(f"Saved LLM-assisted annotation file: {OUTPUT_PATH} ({len(final):,} rows)")

## Inspect Output

In [ ]:
if OUTPUT_PATH.exists():
    llm_df = pd.read_csv(OUTPUT_PATH)
    print(f"Rows with suggestions: {len(llm_df):,}")
    display(llm_df["llm_label_suggestion"].value_counts(dropna=False).to_frame("rows"))
    display(llm_df.head())
else:
    print("No output file yet.")